# Chapter 11: Chatbot Project - Personal Assistant Chatbot

In this chapter, we develop a personal assistant chatbot from scratch. Personal assistant chatbots help users manage daily tasks, answer questions, provide reminders, and engage in casual conversation. They can be integrated into mobile apps, websites, and messaging services.

This project covers the entire development process:
1. **Project Introduction and Design** - Overview, design considerations, system architecture
2. **Data Collection and Preprocessing** - Collecting data, building the NLP engine, handling imbalanced data
3. **Building and Training the Chatbot** - Core functionality, external APIs, task management, interface
4. **Evaluating and Deploying the Chatbot** - Accuracy metrics, response quality, Flask deployment
5. **Improving and Maintaining the Chatbot** - User feedback, retraining, new features, monitoring

## Install Required Packages

In [ ]:
# Install required packages
!pip install tensorflow scikit-learn nltk imbalanced-learn flask requests numpy

In [ ]:
# Download required NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

---
## 11.1 Project Introduction and Design

### 11.1.1 Project Overview

The goal of this project is to create a personal assistant chatbot that can:
- Answer general knowledge questions
- Set reminders and alarms
- Provide weather updates
- Manage to-do lists
- Engage in casual conversation

We combine **rule-based logic** with **self-learning capabilities**, leveraging APIs and NLP techniques. We use Python with TensorFlow, NLTK, and scikit-learn to build and train the chatbot.

### 11.1.2 Design Considerations

Designing a personal assistant chatbot involves several key considerations:

1. **User Experience**: The chatbot should understand natural language inputs and respond appropriately.
2. **Scalability**: The design should allow easy addition of new features and tasks.
3. **Security and Privacy**: User data must be protected and handled securely.
4. **Performance**: The chatbot should be responsive and handle multiple user requests simultaneously.

### 11.1.3 System Architecture

The system architecture consists of five components:

1. **Frontend Interface** - The user interacts through a web or mobile app
2. **NLP Engine** - Processes user inputs, performs NLU, and generates responses
3. **Task Manager** - Handles reminders, to-do lists, and weather fetching
4. **External APIs** - Weather information, knowledge bases, news, etc.
5. **Database** - Stores user data, preferences, and task-related information

```
User Input --> Frontend Interface --> NLP Engine --> Task Manager --> Response
                                         |               |
                                    External APIs     Database
```

### 11.1.4 Implementation Plan

1. Define intents and entities
2. Build the NLP engine
3. Integrate external APIs
4. Develop task management functions
5. Create the frontend interface
6. Test and deploy

### 11.1.5 Project Structure

The recommended project directory structure:

```
personal_assistant_chatbot/
├── data/
│   ├── intents.json          # Predefined intents and entities
│   └── reminders.json        # User reminders and to-do items
├── models/
│   ├── nlp_model.h5          # Trained NLP model
│   └── tokenizer.pickle      # Saved TF-IDF vectorizer
├── scripts/
│   ├── nlp_engine.py         # NLP tasks (intent recognition, entity extraction)
│   ├── task_manager.py       # Task management (reminders, weather)
│   ├── api_integration.py    # External API integration
│   └── chatbot_interface.py  # Frontend interface
├── app.py                    # Main application
└── requirements.txt          # Dependencies
```

### 11.1.6 Defining Intents and Entities

**Intents** represent the user's goals or tasks. **Entities** are the key pieces of information needed to complete those tasks. Below we define the `intents.json` data structure that drives the chatbot.

In [ ]:
import json
import os

# Define the intents data
intents_data = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["Hi", "Hello", "Hey", "Good morning", "Good afternoon", "Good evening"],
            "responses": [
                "Hello! How can I assist you today?",
                "Hi there! What can I do for you?",
                "Hey! How can I help?",
                "Good day! How can I assist you?"
            ]
        },
        {
            "tag": "goodbye",
            "patterns": ["Bye", "Goodbye", "See you later", "Talk to you soon"],
            "responses": [
                "Goodbye! Have a great day!",
                "See you later! Take care!",
                "Talk to you soon!"
            ]
        },
        {
            "tag": "weather",
            "patterns": [
                "What's the weather like?",
                "Tell me the weather",
                "How's the weather today?",
                "Weather update",
                "Is it going to rain today?"
            ],
            "responses": [
                "Let me check the weather for you.",
                "Fetching the weather details...",
                "Here's the weather update."
            ]
        },
        {
            "tag": "reminder",
            "patterns": [
                "Set a reminder",
                "Remind me to",
                "Add a reminder",
                "Can you remind me?",
                "Create a reminder"
            ],
            "responses": [
                "Sure, what would you like to be reminded about?",
                "When would you like the reminder to be set?",
                "Got it. What is the reminder for?"
            ]
        },
        {
            "tag": "knowledge",
            "patterns": [
                "Tell me a fact",
                "Do you know any interesting facts?",
                "Share some knowledge",
                "Give me a fun fact"
            ],
            "responses": [
                "Here's a fun fact: Did you know that honey never spoils?",
                "Sure! Did you know that octopuses have three hearts?",
                "Did you know that a day on Venus is longer than a year on Venus?"
            ]
        }
    ]
}

# Create data directory and save intents file
os.makedirs('data', exist_ok=True)
with open('data/intents.json', 'w') as f:
    json.dump(intents_data, f, indent=4)

print("intents.json created successfully!")
print(f"Number of intents: {len(intents_data['intents'])}")
for intent in intents_data['intents']:
    print(f"  - {intent['tag']}: {len(intent['patterns'])} patterns, {len(intent['responses'])} responses")

---
## 11.2 Data Collection and Preprocessing

Data collection and preprocessing are critical steps in building an effective chatbot. The quality and relevance of the data used to train the model directly impact the chatbot's performance.

### 11.2.1 Collecting Data

For our personal assistant chatbot, we need data covering a wide range of user intents and entities. Sources include:

1. **Manual Data Collection** - Manually create common user queries and responses
2. **Public Datasets** - Cornell Movie Dialogs Corpus, ChatterBot dataset, etc.
3. **API Documentation** - Understand data formats for weather, reminders, etc.

### 11.2.2 Building the NLP Engine

The NLP engine processes user inputs, recognizes intents, and extracts entities. We use TF-IDF vectorization to convert text into numerical features, then train a neural network with TensorFlow for intent classification.

**Pipeline:**
1. Load intents from JSON
2. Extract patterns and tags
3. Encode tags with `LabelEncoder`
4. Vectorize patterns with `TfidfVectorizer`
5. Train a neural network for classification

In [ ]:
import json
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Load the intents file
with open('data/intents.json') as file:
    intents = json.load(file)

# Extract patterns and corresponding tags
patterns = []
tags = []
for intent in intents['intents']:
    for pattern in intent['patterns']:
        patterns.append(pattern)
        tags.append(intent['tag'])

print(f"Total patterns: {len(patterns)}")
print(f"Unique tags: {set(tags)}")
print(f"\nSample patterns and tags:")
for p, t in zip(patterns[:5], tags[:5]):
    print(f"  '{p}' -> {t}")

In [ ]:
# Encode the tags into numerical labels
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(tags)

print("Label encoding:")
for cls, idx in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {cls} -> {idx}")

# Vectorize the patterns using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns).toarray()
y = np.array(labels)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Build the neural network model
model = Sequential([
    # Input layer with 128 neurons and ReLU activation
    Dense(128, input_shape=(X_train.shape[1],), activation='relu'),
    # Dropout layer (50%) to prevent overfitting
    Dropout(0.5),
    # Hidden layer with 64 neurons and ReLU activation
    Dense(64, activation='relu'),
    # Another dropout layer
    Dropout(0.5),
    # Output layer with softmax for multi-class classification
    Dense(len(label_encoder.classes_), activation='softmax')
])

# Compile the model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    verbose=1,
    validation_data=(X_test, y_test)
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import pickle

# Save the model, vectorizer, and label encoder
# Note: pickle is used here because scikit-learn objects (TfidfVectorizer,
# LabelEncoder) cannot be serialized with JSON. This is standard practice
# for ML model persistence.
os.makedirs('models', exist_ok=True)

model.save('models/nlp_model.h5')
print("Model saved to models/nlp_model.h5")

with open('models/vectorizer.pickle', 'wb') as file:
    pickle.dump(vectorizer, file)
print("Vectorizer saved to models/vectorizer.pickle")

with open('models/label_encoder.pickle', 'wb') as file:
    pickle.dump(label_encoder, file)
print("Label encoder saved to models/label_encoder.pickle")

### 11.2.3 Handling Missing or Imbalanced Data

In real-world applications, data may be missing or imbalanced. Addressing these issues during preprocessing is critical for model performance.

- **Missing Data**: Replace missing values with a placeholder (mean/median) or remove instances containing missing data.
- **Imbalanced Data**: Use oversampling (SMOTE), undersampling, or synthetic sample generation to balance the dataset.

In [ ]:
from imblearn.over_sampling import SMOTE

# Check for missing data
print(f"Missing values in X: {np.isnan(X).sum()}")

# Handle missing data (replace NaN with 0)
X_clean = np.nan_to_num(X)

# Check class distribution before SMOTE
unique, counts = np.unique(y, return_counts=True)
print(f"\nClass distribution before SMOTE:")
for u, c in zip(unique, counts):
    print(f"  Class {label_encoder.classes_[u]}: {c} samples")

# Balance the dataset using SMOTE
# Note: SMOTE requires at least k_neighbors+1 samples per class
# We use k_neighbors=1 since our dataset is small
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_clean, y)

# Check class distribution after SMOTE
unique_r, counts_r = np.unique(y_resampled, return_counts=True)
print(f"\nClass distribution after SMOTE:")
for u, c in zip(unique_r, counts_r):
    print(f"  Class {label_encoder.classes_[u]}: {c} samples")

print(f"\nDataset size: {len(y)} -> {len(y_resampled)} (after SMOTE)")

In [ ]:
# Retrain with balanced dataset
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Build a fresh model for the balanced data
model_balanced = Sequential([
    Dense(128, input_shape=(X_train_bal.shape[1],), activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model_balanced.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_bal = model_balanced.fit(
    X_train_bal, y_train_bal,
    epochs=100,
    batch_size=8,
    verbose=0,
    validation_data=(X_test_bal, y_test_bal)
)

print(f"Balanced model - Final training accuracy: {history_bal.history['accuracy'][-1]:.4f}")
print(f"Balanced model - Final validation accuracy: {history_bal.history['val_accuracy'][-1]:.4f}")

---
## 11.3 Building and Training the Chatbot

In this section, we implement the core functionalities: the NLP engine for intent recognition, task management functions, integration with external APIs, and the chatbot interface.

### 11.3.1 Implementing the Core Functionality - NLP Engine

The NLP engine loads the trained model and uses it to predict user intent. It includes text preprocessing (lowercasing, tokenization, stop word removal, lemmatization) before making predictions.

In [ ]:
import json
import numpy as np
import pickle
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model

# Load the pre-trained model and resources
nlp_model = load_model('models/nlp_model.h5')

with open('models/vectorizer.pickle', 'rb') as file:
    nlp_vectorizer = pickle.load(file)

with open('models/label_encoder.pickle', 'rb') as file:
    nlp_label_encoder = pickle.load(file)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """Preprocess text: lowercase, tokenize, remove stopwords/punctuation, lemmatize."""
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [
        word for word in tokens
        if word not in string.punctuation and word not in stopwords.words('english')
    ]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)


def predict_intent(text):
    """Predict the intent of the given text using the trained model."""
    preprocessed_text = preprocess_text(text)
    input_vector = nlp_vectorizer.transform([preprocessed_text]).toarray()
    predictions = nlp_model.predict(input_vector, verbose=0)
    predicted_intent = nlp_label_encoder.inverse_transform([np.argmax(predictions)])
    return predicted_intent[0]


# Test the NLP engine
test_inputs = [
    "Hello there!",
    "Can you set a reminder?",
    "What's the weather like?",
    "Tell me a fun fact",
    "Goodbye!"
]

print("NLP Engine - Intent Prediction Tests:")
print("-" * 50)
for test_input in test_inputs:
    intent = predict_intent(test_input)
    print(f"  Input: '{test_input}' -> Intent: {intent}")

### 11.3.2 Integrating External APIs

To enhance the chatbot's functionality, we integrate external APIs. Below is an example using the OpenWeatherMap API for weather updates.

> **Note**: Replace `'your_openweathermap_api_key'` with a real API key from [OpenWeatherMap](https://openweathermap.org/api) to make the weather function work. The demo below includes a fallback for when no API key is available.

In [ ]:
import requests


def get_weather(city, api_key="your_openweathermap_api_key"):
    """
    Fetch weather information for a given city using OpenWeatherMap API.
    Returns a formatted string with temperature and weather description.
    """
    base_url = "http://api.openweathermap.org/data/2.5/weather?"
    complete_url = base_url + "q=" + city + "&appid=" + api_key + "&units=metric"

    try:
        response = requests.get(complete_url, timeout=5)
        data = response.json()

        if data.get("cod") != "404" and "main" in data:
            main = data["main"]
            weather = data["weather"][0]
            temperature = main["temp"]
            weather_description = weather["description"]
            return f"The temperature in {city} is {temperature} degrees C with {weather_description}."
        else:
            return f"City '{city}' not found or API error."
    except requests.exceptions.RequestException:
        return f"[Demo mode] Weather for {city}: 22 degrees C with partly cloudy skies."


# Test the weather API integration
city = "New York"
weather_update = get_weather(city)
print(f"Weather Update: {weather_update}")

### 11.3.3 Implementing Task Management Functions

The task manager handles specific tasks like setting reminders and managing to-do lists. Reminders are stored in a JSON file for persistence.

In [ ]:
import json
from datetime import datetime


def load_reminders(filepath='data/reminders.json'):
    """Load reminders from a JSON file."""
    try:
        with open(filepath, 'r') as file:
            reminders = json.load(file)
    except FileNotFoundError:
        reminders = []
    return reminders


def save_reminders(reminders, filepath='data/reminders.json'):
    """Save reminders to a JSON file."""
    with open(filepath, 'w') as file:
        json.dump(reminders, file, indent=4)


def add_reminder(reminder_text, reminder_time):
    """Add a new reminder."""
    reminders = load_reminders()
    reminder = {
        "text": reminder_text,
        "time": reminder_time
    }
    reminders.append(reminder)
    save_reminders(reminders)
    return "Reminder added successfully."


def view_reminders():
    """View all reminders."""
    reminders = load_reminders()
    if not reminders:
        return "You have no reminders."
    reminders_str = "\n".join(
        [f"  {reminder['time']}: {reminder['text']}" for reminder in reminders]
    )
    return f"Your reminders:\n{reminders_str}"


def delete_reminder(reminder_text):
    """Delete a reminder by its text."""
    reminders = load_reminders()
    reminders = [r for r in reminders if r['text'] != reminder_text]
    save_reminders(reminders)
    return "Reminder deleted successfully."


# Test the task manager functions
print(add_reminder("Buy groceries", "2024-06-27 18:00"))
print(add_reminder("Call dentist", "2024-06-28 10:00"))
print()
print(view_reminders())
print()
print(delete_reminder("Buy groceries"))
print()
print(view_reminders())

### 11.3.4 Building the Chatbot Interface

The chatbot interface ties all components together. It takes user input, predicts the intent using the NLP engine, and routes to the appropriate handler (weather API, task manager, etc.).

Below is a programmatic (non-interactive) demonstration. In a real application, this would run as an interactive loop or as a Flask web endpoint.

In [ ]:
import random

# Load intents for response selection
with open('data/intents.json') as f:
    chatbot_intents = json.load(f)

# Build a response lookup from intents
intent_responses = {}
for intent in chatbot_intents['intents']:
    intent_responses[intent['tag']] = intent['responses']


def chatbot_response(user_input):
    """
    Generate a chatbot response based on the predicted intent.
    Routes to appropriate handlers for specific tasks.
    """
    intent = predict_intent(user_input)

    if intent == "greeting":
        return random.choice(intent_responses.get('greeting', ["Hello!"]))
    elif intent == "goodbye":
        return random.choice(intent_responses.get('goodbye', ["Goodbye!"]))
    elif intent == "weather":
        return "Which city would you like the weather update for?"
    elif intent == "reminder":
        return "What would you like to be reminded about and when?"
    elif intent == "knowledge":
        return random.choice(intent_responses.get('knowledge', ["Here's a fun fact: Honey never spoils!"]))
    else:
        return "I'm sorry, I don't understand that. Can you please rephrase?"


# Simulate a conversation
test_conversation = [
    "Hello!",
    "What's the weather like today?",
    "Can you set a reminder for me?",
    "Tell me something interesting",
    "Goodbye!"
]

print("Simulated Conversation:")
print("=" * 50)
for msg in test_conversation:
    response = chatbot_response(msg)
    print(f"You: {msg}")
    print(f"Bot: {response}")
    print()

---
## 11.4 Evaluating and Deploying the Chatbot

Evaluation ensures the chatbot performs well in real-world scenarios. We assess accuracy, response quality, and response time. Deployment makes it accessible to users.

### 11.4.1 Evaluating the Chatbot

#### Accuracy Metrics

We use precision, recall, and F1-score to measure how correctly the chatbot identifies intents.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Print classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(
    y_test, y_pred_classes,
    target_names=label_encoder.classes_,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Intent Classification - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

#### Response Time

Response time is critical for user experience. We measure how quickly the chatbot processes and responds to queries.

In [ ]:
import time


def measure_response_time(user_input):
    """Measure the response time for a given user input."""
    start_time = time.time()
    response = chatbot_response(user_input)
    end_time = time.time()
    response_time = end_time - start_time
    return response, response_time


# Test response times for various queries
test_queries = [
    "Hello!",
    "What's the weather like?",
    "Set a reminder",
    "Tell me a fact",
    "Goodbye"
]

print("Response Time Measurements:")
print("-" * 60)
total_time = 0
for query in test_queries:
    response, resp_time = measure_response_time(query)
    total_time += resp_time
    print(f"  Query: '{query}'")
    print(f"  Response: '{response}'")
    print(f"  Time: {resp_time:.4f} seconds\n")

avg_time = total_time / len(test_queries)
print(f"Average response time: {avg_time:.4f} seconds")

### 11.4.2 Deploying the Chatbot

#### Web Application Deployment with Flask

Flask is a lightweight web framework ideal for serving the chatbot. The application exposes a `/chat` endpoint that accepts POST requests with user messages and returns chatbot responses.

> **Note**: The Flask app below is shown as a code reference. To run it, save it as `app.py` and execute it from the command line with `python app.py`.

In [ ]:
# Flask application code (reference - save as app.py to run)
flask_app_code = '''
from flask import Flask, request, jsonify
from nlp_engine import predict_intent
from api_integration import get_weather
from task_manager import add_reminder, view_reminders, delete_reminder

app = Flask(__name__)

@app.route('/chat', methods=['POST'])
def chat():
    user_input = request.json.get('message')
    intent = predict_intent(user_input)

    if intent == "greeting":
        response = "Hello! How can I assist you today?"
    elif intent == "goodbye":
        response = "Goodbye! Have a great day!"
    elif intent == "weather":
        city = request.json.get('city', 'New York')
        response = get_weather(city)
    elif intent == "reminder":
        reminder_text = request.json.get('reminder_text')
        reminder_time = request.json.get('reminder_time')
        response = add_reminder(reminder_text, reminder_time)
    elif intent == "view_reminders":
        response = view_reminders()
    elif intent == "delete_reminder":
        reminder_text = request.json.get('reminder_text')
        response = delete_reminder(reminder_text)
    else:
        response = "I'm sorry, I don't understand that. Can you please rephrase?"

    return jsonify({'response': response})

if __name__ == '__main__':
    app.run(debug=True)
'''

print("Flask Application Code (save as app.py):")
print("=" * 50)
print(flask_app_code)
print("To run: python app.py")
print("Then send POST requests to http://localhost:5000/chat")
print('Example: curl -X POST http://localhost:5000/chat -H "Content-Type: application/json" -d \'{"message": "Hello"}\'')

#### Integrating with Messaging Apps (Facebook Messenger)

The chatbot can also be integrated with messaging platforms. Below is a reference implementation for Facebook Messenger webhook integration:

**Steps:**
1. Create a Facebook Page for your chatbot
2. Set up a Facebook Developer account
3. Create a Facebook App and add the Messenger product
4. Generate a Page Access Token
5. Set up a webhook to receive messages

In [ ]:
# Facebook Messenger Webhook (reference implementation)
messenger_code = '''
from flask import Flask, request
import requests

app = Flask(__name__)
PAGE_ACCESS_TOKEN = 'your_facebook_page_access_token'

def send_message(recipient_id, message_text):
    """Send a message to a Facebook Messenger user."""
    url = f"https://graph.facebook.com/v11.0/me/messages?access_token={PAGE_ACCESS_TOKEN}"
    headers = {"Content-Type": "application/json"}
    payload = {
        "recipient": {"id": recipient_id},
        "message": {"text": message_text}
    }
    requests.post(url, headers=headers, json=payload)

@app.route('/webhook', methods=['GET'])
def webhook_verification():
    """Verify the webhook with Facebook."""
    mode = request.args.get('hub.mode')
    token = request.args.get('hub.verify_token')
    challenge = request.args.get('hub.challenge')
    if mode == 'subscribe' and token == 'your_verify_token':
        return challenge
    return 'Verification failed', 403

@app.route('/webhook', methods=['POST'])
def webhook():
    """Handle incoming messages from Facebook Messenger."""
    data = request.json
    if data['object'] == 'page':
        for entry in data['entry']:
            for messaging_event in entry['messaging']:
                sender_id = messaging_event['sender']['id']
                if 'message' in messaging_event:
                    message_text = messaging_event['message']['text']
                    response_text = chatbot_response(message_text)
                    send_message(sender_id, response_text)
    return 'OK', 200

if __name__ == '__main__':
    app.run(debug=True)
'''

print("Facebook Messenger Webhook Code:")
print("=" * 50)
print(messenger_code)

---
## 11.5 Improving and Maintaining the Chatbot

Building a chatbot is an iterative process that requires continuous improvement and maintenance. This includes collecting user feedback, retraining the model, adding new features, and monitoring performance.

### 11.5.1 Collecting User Feedback

User feedback is invaluable for understanding how well the chatbot meets user needs. We add a `/feedback` endpoint where users can submit ratings and comments.

In [ ]:
# Feedback collection system
feedback_data = []


def collect_feedback(message, response, rating, comment=""):
    """Collect user feedback on chatbot interactions."""
    feedback = {
        "message": message,
        "response": response,
        "rating": rating,  # 1-5 scale
        "comment": comment,
        "timestamp": datetime.now().isoformat()
    }
    feedback_data.append(feedback)
    return "Thank you for your feedback!"


def analyze_feedback():
    """Analyze collected feedback."""
    if not feedback_data:
        return "No feedback collected yet."

    ratings = [f['rating'] for f in feedback_data]
    avg_rating = sum(ratings) / len(ratings)
    low_ratings = [f for f in feedback_data if f['rating'] <= 2]

    report = f"Feedback Summary:\n"
    report += f"  Total feedback: {len(feedback_data)}\n"
    report += f"  Average rating: {avg_rating:.2f}/5\n"
    report += f"  Low-rated interactions: {len(low_ratings)}\n"

    if low_ratings:
        report += "\n  Low-rated interactions to review:\n"
        for f in low_ratings:
            report += f"    - '{f['message']}' (rating: {f['rating']}) - {f['comment']}\n"

    return report


# Simulate feedback collection
print(collect_feedback(
    "What's the weather like?",
    "The temperature in New York is 25 degrees C with clear skies.",
    rating=4,
    comment="The response was quick and accurate."
))

print(collect_feedback(
    "Book a flight for me",
    "I'm sorry, I don't understand that. Can you please rephrase?",
    rating=2,
    comment="Chatbot should handle travel requests."
))

print(collect_feedback(
    "Hello!",
    "Hello! How can I assist you today?",
    rating=5,
    comment="Friendly greeting."
))

print()
print(analyze_feedback())

### 11.5.2 Retraining the Model

As we collect more data and feedback, we retrain the model to improve accuracy. This involves updating the dataset with new patterns, preprocessing the new data, and retraining with the expanded dataset.

In [ ]:
import json
import numpy as np
import pickle
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


def retrain_model(intents_path='data/intents.json'):
    """
    Retrain the intent classification model with updated intents data.
    Handles the full pipeline: load, preprocess, train, save.
    """
    # Load updated intents
    with open(intents_path) as file:
        intents = json.load(file)

    # Initialize lemmatizer
    lem = WordNetLemmatizer()

    def preprocess(text):
        text = text.lower()
        tokens = nltk.word_tokenize(text)
        tokens = [
            word for word in tokens
            if word not in string.punctuation and word not in stopwords.words('english')
        ]
        tokens = [lem.lemmatize(word) for word in tokens]
        return ' '.join(tokens)

    # Preprocess patterns and extract tags
    patterns = []
    tags = []
    for intent in intents['intents']:
        for pattern in intent['patterns']:
            patterns.append(preprocess(pattern))
            tags.append(intent['tag'])

    # Encode and vectorize
    le = LabelEncoder()
    labels = le.fit_transform(tags)

    vec = TfidfVectorizer()
    X = vec.fit_transform(patterns).toarray()
    y = np.array(labels)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Build and train model
    new_model = Sequential([
        Dense(128, input_shape=(X_train.shape[1],), activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(len(le.classes_), activation='softmax')
    ])

    new_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    history = new_model.fit(X_train, y_train, epochs=100, batch_size=8, verbose=0,
                           validation_data=(X_test, y_test))

    # Save everything
    new_model.save('models/nlp_model.h5')
    with open('models/vectorizer.pickle', 'wb') as f:
        pickle.dump(vec, f)
    with open('models/label_encoder.pickle', 'wb') as f:
        pickle.dump(le, f)

    final_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    return final_acc, final_val_acc


# Demonstrate retraining with the current intents data
print("Retraining model with current intents data...")
train_acc, val_acc = retrain_model()
print(f"Retraining complete!")
print(f"  Training accuracy: {train_acc:.4f}")
print(f"  Validation accuracy: {val_acc:.4f}")

### 11.5.3 Adding New Features

To keep the chatbot relevant, continuously add new features based on user feedback. Below we add a **news update** feature using the News API.

In [ ]:
import requests


def get_news(api_key="your_newsapi_api_key"):
    """
    Fetch top news headlines using the News API.
    Replace api_key with a real key from https://newsapi.org/
    """
    url = f"https://newsapi.org/v2/top-headlines?country=us&apiKey={api_key}"

    try:
        response = requests.get(url, timeout=5)
        data = response.json()

        if data.get("status") == "ok":
            articles = data["articles"]
            news_list = [
                f"{article['title']} - {article['source']['name']}"
                for article in articles[:5]
            ]
            return "Here are the top news headlines:\n" + "\n".join(news_list)
        else:
            return "Unable to fetch news at the moment."
    except requests.exceptions.RequestException:
        # Demo fallback
        return (
            "[Demo mode] Here are the top news headlines:\n"
            "1. Tech Giants Report Strong Q4 Earnings - Reuters\n"
            "2. Climate Summit Reaches Historic Agreement - BBC\n"
            "3. New AI Breakthrough in Medical Diagnosis - CNN"
        )


# Test the news feature
print(get_news())

In [ ]:
# Enhanced chatbot response with news feature
def chatbot_response_v2(user_input):
    """
    Enhanced chatbot response that includes the news feature.
    """
    intent = predict_intent(user_input)

    if intent == "greeting":
        return random.choice(intent_responses.get('greeting', ["Hello!"]))
    elif intent == "goodbye":
        return random.choice(intent_responses.get('goodbye', ["Goodbye!"]))
    elif intent == "weather":
        return "Which city would you like the weather update for?"
    elif intent == "reminder":
        return "What would you like to be reminded about and when?"
    elif intent == "knowledge":
        return random.choice(intent_responses.get('knowledge', ["Here's a fun fact!"]))
    else:
        # Check for news-related keywords as a fallback
        lower_input = user_input.lower()
        if any(word in lower_input for word in ['news', 'headlines', 'latest']):
            return get_news()
        return "I'm sorry, I don't understand that. Can you please rephrase?"


# Test the enhanced chatbot
print("Enhanced Chatbot Tests:")
print("=" * 50)
for query in ["Hi!", "Give me the latest news", "Bye!"]:
    resp = chatbot_response_v2(query)
    print(f"You: {query}")
    print(f"Bot: {resp}")
    print()

### 11.5.4 Monitoring and Maintenance

Regular monitoring is essential to ensure the chatbot continues to perform well. Key activities:

- **Logging user interactions** for analysis
- **Tracking response times** and accuracy
- **Regular updates** with new data and features
- **Performance dashboards** to visualize trends

In [ ]:
import logging
from datetime import datetime
from collections import Counter

# Simulated interaction log
interaction_log = []


def log_interaction(user_input, predicted_intent, response, response_time):
    """Log a chatbot interaction for monitoring."""
    entry = {
        "timestamp": datetime.now().isoformat(),
        "user_input": user_input,
        "predicted_intent": predicted_intent,
        "response": response,
        "response_time": response_time
    }
    interaction_log.append(entry)
    return entry


def chatbot_with_logging(user_input):
    """Chatbot response with interaction logging."""
    start_time = time.time()
    intent = predict_intent(user_input)
    response = chatbot_response(user_input)
    response_time = time.time() - start_time

    log_interaction(user_input, intent, response, response_time)
    return response


# Run several interactions through the logged chatbot
test_inputs = [
    "Hello!", "What's the weather?", "Set a reminder",
    "Tell me something interesting", "Goodbye!",
    "Hi there", "How's the weather today?"
]

for inp in test_inputs:
    chatbot_with_logging(inp)

# Analyze the interaction log
print("Interaction Log Analysis:")
print("=" * 50)
print(f"Total interactions: {len(interaction_log)}")

# Response time statistics
times = [entry['response_time'] for entry in interaction_log]
print(f"Average response time: {np.mean(times):.4f}s")
print(f"Max response time: {np.max(times):.4f}s")
print(f"Min response time: {np.min(times):.4f}s")

# Intent distribution
intent_counts = Counter(entry['predicted_intent'] for entry in interaction_log)
print(f"\nIntent distribution:")
for intent, count in intent_counts.most_common():
    print(f"  {intent}: {count} ({count/len(interaction_log)*100:.1f}%)")

In [ ]:
# Visualize monitoring data
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Intent distribution pie chart
labels_pie = list(intent_counts.keys())
sizes = list(intent_counts.values())
axes[0].pie(sizes, labels=labels_pie, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Intent Distribution')

# Response time bar chart
queries_short = [entry['user_input'][:15] + '...' if len(entry['user_input']) > 15
                 else entry['user_input'] for entry in interaction_log]
axes[1].barh(queries_short, times, color='steelblue')
axes[1].set_xlabel('Response Time (seconds)')
axes[1].set_title('Response Times per Query')

plt.tight_layout()
plt.show()

---
## Chapter Summary

In this chapter, we developed a personal assistant chatbot through the full project lifecycle:

**Project Introduction and Design**: We outlined the project goals (answering questions, setting reminders, weather updates, to-do lists, casual conversation) and the system architecture (frontend, NLP engine, task manager, external APIs, database).

**Data Collection and Preprocessing**: We built an intents dataset with patterns and responses, vectorized text with TF-IDF, and addressed imbalanced data using SMOTE.

**Building and Training the Chatbot**: We implemented the NLP engine with TensorFlow for intent recognition, integrated external APIs (weather, news), developed task management (reminders), and built the chatbot interface.

**Evaluating and Deploying**: We evaluated the model with precision/recall/F1-score, measured response times, and demonstrated deployment with Flask and Facebook Messenger integration.

**Improving and Maintaining**: We covered user feedback collection, model retraining with updated data, adding new features (news), and monitoring through interaction logging and visualization.

Through continuous improvement and maintenance, this chatbot can be built into a robust, adaptive assistant that enhances user productivity.

In [ ]:
# Cleanup: remove temporary data files created during this notebook
import shutil

for folder in ['data', 'models']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up '{folder}/' directory")

print("\nNotebook complete!")